# Malware Traffic Classification Using Machine Learning

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Dataset URL
data_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"

# Manually defined columns
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent',
    'hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
    'dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','label'
]

# Load dataset
df = pd.read_csv(data_url, names=columns)

# One-hot encode the 'label' column
df['label'] = df['label'].apply(lambda x: 'normal' if x == 'normal' else 'malicious')
df = pd.get_dummies(df)

# Print out the columns after one-hot encoding to check
print("Columns after get_dummies:", df.columns)

# Drop the label column (after one-hot encoding) and separate features and target
X = df.drop('label_malicious', axis=1)  # Drop the target column from the features
y = df['label_malicious']  # Set the target column

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
clf = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=666)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='inferno', xticklabels=['normal', 'malicious'], yticklabels=['normal', 'malicious'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature importance
importances = clf.feature_importances_
indices = np.argsort(importances)[-10:][::-1]
top_features = [X.columns[i] for i in indices]

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=top_features, palette="rocket")
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()
